# RAG with OpenAI — Native Implementation (Local PDFs)

**RAG = Retrieval-Augmented Generation**

Instead of asking the AI from memory, we:
1. Read PDF files from a local folder
2. Split them into chunks and convert each chunk into an *embedding* (numbers that capture meaning)
3. When a question comes in, find the most relevant chunks
4. Hand those chunks to GPT so it can answer with real facts from your documents

```
📁 pdfs/
  └── doc1.pdf, doc2.pdf, ...   ← your files go here
        ↓
  Extract text → Chunk → Embed → Vector Store
                                       ↓
  User Question → Embed → Search → Top Chunks → GPT → Answer
```

---
## Step 1 — Install & Import

Install the packages we need, then load them.  
`pypdf` is used to read text out of PDF files.

In [ ]:
# Install required libraries (run once)
!pip install openai pypdf tiktoken python-dotenv --quiet

In [ ]:
import os
import json
import math
from pathlib import Path
from typing import List, Dict

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import tiktoken

# Load API key from .env file  ← create a .env file with: OPENAI_API_KEY=sk-...
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Setup complete. OpenAI client ready.")

---
## Step 2 — Point to Your PDF Folder

Set `PDF_FOLDER` to the folder that contains your `.pdf` files.  
The folder is created automatically if it doesn't exist yet —  
just drop your PDFs in there and re-run this cell.

In [ ]:
# ── Configure this path ──────────────────────────────────────────────────
PDF_FOLDER = Path("pdfs")   # relative to the notebook; change if needed
# e.g. PDF_FOLDER = Path("/Users/you/Documents/my_pdfs")
# ────────────────────────────────────────────────────────────────────────

PDF_FOLDER.mkdir(exist_ok=True)  # create the folder if it doesn't exist

pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

if not pdf_files:
    print(f"⚠️  No PDF files found in '{PDF_FOLDER.resolve()}'.")
    print("   Please add at least one .pdf file to that folder and re-run this cell.")
else:
    print(f"✓ Found {len(pdf_files)} PDF file(s) in '{PDF_FOLDER.resolve()}':")
    for p in pdf_files:
        print(f"   📄 {p.name}")

---
## Step 3 — Extract Text from PDFs

`pypdf` opens each file, reads every page, and concatenates the text.  
Each PDF becomes one document dict with a `source` name and its full `text`.

> **Tip:** If a PDF is scanned (images only), `pypdf` won't extract text.  
> In that case you'd need an OCR tool like `pytesseract`. A warning is printed below if this happens.

In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Open a PDF and return all its text as a single string.
    Pages are joined with a newline. Empty/image-only pages are skipped.
    """
    reader = PdfReader(str(pdf_path))
    pages_text = []

    for page in reader.pages:
        text = (page.extract_text() or "").strip()
        if text:
            pages_text.append(text)

    return "\n".join(pages_text)


# ── Load all PDFs into document dicts ────────────────────────────────────
DOCUMENTS: List[Dict] = []
skipped = []

for pdf_path in pdf_files:
    text = extract_text_from_pdf(pdf_path)

    if not text:
        print(f"  ⚠️  '{pdf_path.name}' — no text extracted (possibly scanned). Skipping.")
        skipped.append(pdf_path.name)
        continue

    DOCUMENTS.append({"source": pdf_path.name, "text": text})
    print(f"  ✓ '{pdf_path.name}' — {len(text.split()):,} words extracted")

print(f"\n✓ Loaded {len(DOCUMENTS)} document(s) successfully.")
if skipped:
    print(f"  Skipped (no text): {skipped}")

---
## Step 4 — Chunk the Documents

**Why chunk?** Models have context limits and retrieval works better on focused passages.  
We split each document into overlapping windows measured in *tokens* (not characters),  
because that's how OpenAI's embedding model measures length.

- `CHUNK_SIZE = 200` tokens ≈ ~150 words per chunk  
- `CHUNK_OVERLAP = 40` tokens — the tail of one chunk repeats in the next so sentences aren't cut off

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"   # used for both chunking and embedding
CHUNK_SIZE      = 200   # tokens per chunk
CHUNK_OVERLAP   = 40    # overlap between consecutive chunks


def chunk_text(
    text: str,
    source: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
    model: str = EMBEDDING_MODEL,
) -> List[Dict]:
    """
    Split text into overlapping token-based chunks.

    Steps:
    1. Tokenize the full text with tiktoken.
    2. Slide a window of `chunk_size` tokens, stepping by (chunk_size - overlap).
    3. Decode each window back to a string.
    4. Return a list of chunk dicts with source, index, text, and token count.
    """
    enc    = tiktoken.encoding_for_model(model)
    tokens = enc.encode(text)
    step   = chunk_size - overlap

    chunks = []
    for idx, start in enumerate(range(0, len(tokens), step)):
        token_slice = tokens[start : start + chunk_size]
        if not token_slice:
            break
        chunks.append({
            "source":      source,
            "chunk_id":    idx,
            "text":        enc.decode(token_slice),
            "token_count": len(token_slice),
        })
        if start + chunk_size >= len(tokens):
            break

    return chunks


# ── Chunk all documents ──────────────────────────────────────────────────
all_chunks: List[Dict] = []

for doc in DOCUMENTS:
    doc_chunks = chunk_text(doc["text"], doc["source"])
    all_chunks.extend(doc_chunks)
    print(f"  '{doc['source']}' → {len(doc_chunks)} chunks")

print(f"\n✓ Total chunks: {len(all_chunks)}")
print("\nFirst chunk preview:")
print(json.dumps({k: v for k, v in all_chunks[0].items() if k != "embedding"}, indent=2))

---
## Step 5 — Generate Embeddings

**What is an embedding?** A list of ~1536 numbers that encodes the *meaning* of a text.  
Similar texts produce similar vectors — that's what lets us do semantic search.

We send chunks to the OpenAI embedding API in **batches of 100** to stay within payload limits.

In [ ]:
def get_embeddings_batch(
    texts: List[str],
    model: str = EMBEDDING_MODEL,
    batch_size: int = 100,
) -> List[List[float]]:
    """
    Send texts to OpenAI in batches and return their embedding vectors.
    Batching avoids payload limits and makes progress easy to track.
    """
    all_embeddings: List[List[float]] = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        # API guarantees items are returned in the same order as input
        all_embeddings.extend(item.embedding for item in response.data)
        processed = min(i + batch_size, len(texts))
        print(f"  Embedded {processed}/{len(texts)} chunks…")

    return all_embeddings


# ── Embed every chunk ────────────────────────────────────────────────────
print("Generating embeddings…")
chunk_texts = [c["text"] for c in all_chunks]
embeddings  = get_embeddings_batch(chunk_texts)

# Attach each embedding vector back to its chunk dict
for chunk, emb in zip(all_chunks, embeddings):
    chunk["embedding"] = emb

print(f"\n✓ Done! {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions.")

---
## Step 6 — Build an In-Memory Vector Store

A **vector store** holds our chunks + embeddings so we can search them quickly.  
We use plain Python here — no external database needed.  
For millions of vectors you'd switch to FAISS, Pinecone, or Weaviate.

In [ ]:
class SimpleVectorStore:
    """
    Minimal in-memory vector store.

    add()    → store chunk dicts that already have an 'embedding' key.
    search() → return the top-k chunks most similar to a query vector.

    Similarity is measured with cosine similarity:
        sim(A, B) = dot(A, B) / (|A| × |B|)
    Score near 1.0 = very similar; near 0 = unrelated.
    """

    def __init__(self):
        self.chunks: List[Dict] = []

    def add(self, chunks: List[Dict]):
        self.chunks.extend(chunks)
        print(f"✓ Vector store now holds {len(self.chunks)} chunks.")

    @staticmethod
    def _cosine_similarity(a: List[float], b: List[float]) -> float:
        dot   = sum(x * y for x, y in zip(a, b))
        mag_a = math.sqrt(sum(x * x for x in a))
        mag_b = math.sqrt(sum(x * x for x in b))
        return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

    def search(self, query_embedding: List[float], top_k: int = 3) -> List[Dict]:
        """Score every chunk against the query and return the top_k results."""
        scored = [
            {**chunk, "score": self._cosine_similarity(query_embedding, chunk["embedding"])}
            for chunk in self.chunks
        ]
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]


# ── Populate the store ───────────────────────────────────────────────────
vector_store = SimpleVectorStore()
vector_store.add(all_chunks)

---
## Step 7 — Retrieval Function

Given a question, embed it with the **same model**, then search the store.  
Using the same model is critical — otherwise the vector spaces won't align.

In [ ]:
def retrieve(
    query: str,
    store: SimpleVectorStore,
    top_k: int = 3,
    model: str = EMBEDDING_MODEL,
) -> List[Dict]:
    """
    1. Embed the user's question.
    2. Find the top_k most similar chunks in the vector store.
    3. Return those chunks (with source, chunk_id, and similarity score).
    """
    response = client.embeddings.create(model=model, input=[query])
    query_embedding = response.data[0].embedding
    return store.search(query_embedding, top_k=top_k)


# ── Quick retrieval test ─────────────────────────────────────────────────
test_query = input("Enter a test question to check retrieval (or press Enter to skip): ").strip()

if test_query:
    test_results = retrieve(test_query, vector_store)
    print(f'\nTop {len(test_results)} chunks for: "{test_query}"\n')
    for i, chunk in enumerate(test_results, 1):
        print(f"[{i}] Score: {chunk['score']:.4f} | {chunk['source']} chunk #{chunk['chunk_id']}")
        print(f"     {chunk['text'][:150].strip()}…\n")
else:
    print("Skipped retrieval test.")

---
## Step 8 — RAG Answer Function

Now we combine retrieval with generation:
1. Retrieve the top relevant chunks from your PDFs
2. Format them as *context* in the prompt (with source labels)
3. Ask GPT to answer **using only that context** and cite every fact

In [ ]:
CHAT_MODEL = "gpt-4o-mini"   # cheap and fast; swap for gpt-4o for higher quality


def format_context(chunks: List[Dict]) -> str:
    """
    Turn retrieved chunks into a readable context block for the prompt.
    Each chunk is labelled with its PDF filename and chunk index.
    """
    parts = []
    for i, chunk in enumerate(chunks, 1):
        label = f"Source {i}: {chunk['source']} (chunk #{chunk['chunk_id']})"
        parts.append(f"[{label}]\n{chunk['text']}")
    return "\n\n".join(parts)


def rag_answer(
    question: str,
    store: SimpleVectorStore,
    top_k: int = 3,
    chat_model: str = CHAT_MODEL,
) -> Dict:
    """
    Full RAG pipeline:
      retrieve → format context → Chat Completions → return answer + sources.

    Returns a dict with:
      'answer'  — the model's response string
      'sources' — list of (filename, chunk_id, similarity_score)
      'chunks'  — the raw retrieved chunks (for inspection / export)
    """
    # A — Retrieve relevant chunks from your PDFs
    chunks  = retrieve(question, store, top_k=top_k)
    context = format_context(chunks)

    # B — Build the prompt
    system_prompt = (
        "You are a helpful assistant. Answer the user's question ONLY using the "
        "provided context extracted from their PDF documents. "
        "If the context doesn't contain the answer, say so clearly — do not guess. "
        "Always cite the source label (e.g. [Source 1]) for every fact you state."
    )

    user_message = (
        f"Context from your PDFs:\n\n{context}\n\n"
        f"Question: {question}"
    )

    # C — Call the Chat Completions API
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=0,   # 0 = deterministic / factual
    )

    answer  = response.choices[0].message.content
    sources = [(c["source"], c["chunk_id"], round(c["score"], 4)) for c in chunks]

    return {"answer": answer, "sources": sources, "chunks": chunks}


print("✓ RAG function ready. Proceed to the next cell to ask questions.")

---
## Step 9 — Interactive Q&A Loop

Ask questions about your PDFs!  
Every answer is grounded in the retrieved passages and cites its source file.  
Type `quit` to stop. All Q&A pairs are stored in `session_results` for export.

In [ ]:
def pretty_print(result: Dict, question: str):
    """Display a RAG result in a clean, readable format."""
    sep = "=" * 70
    print(sep)
    print(f"❓ QUESTION: {question}")
    print(sep)
    print(f"\n💬 ANSWER:\n{result['answer']}")
    print("\n📚 RETRIEVED FROM:")
    for src, chunk_id, score in result["sources"]:
        print(f"   • {src}  chunk #{chunk_id}  (similarity: {score})")
    print()


session_results: List[Dict] = []   # collects all Q&A pairs for export

print("Interactive RAG — ask questions about your PDFs.")
print("Type 'quit' to stop and save results.\n")

while True:
    user_q = input("Your question: ").strip()
    if not user_q or user_q.lower() in ("quit", "exit", "q"):
        print(f"\nSession ended. {len(session_results)} question(s) recorded.")
        break

    res = rag_answer(user_q, vector_store)
    pretty_print(res, user_q)
    session_results.append({"question": user_q, **res})

---
## Step 10 — Save Results to JSON

Export all Q&A pairs (without the large embedding vectors) for your submission.

In [ ]:
if not session_results:
    print("No results to save — ask at least one question in Step 9 first.")
else:
    # Strip large embedding vectors before saving
    export = []
    for r in session_results:
        export.append({
            "question": r["question"],
            "answer":   r["answer"],
            "sources":  r["sources"],
            "top_chunks": [
                {
                    "source":   c["source"],
                    "chunk_id": c["chunk_id"],
                    "score":    round(c["score"], 4),
                    "text":     c["text"],
                }
                for c in r["chunks"]
            ],
        })

    output_path = "rag_results.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(export, f, indent=2, ensure_ascii=False)

    print(f"✓ Results saved to '{output_path}'")
    print(f"  {len(export)} Q&A pair(s) exported.")

---
## Summary — What we built

| Step | What happened | Key tool |
|------|--------------|----------|
| 1 | Installed & imported libraries | `openai`, `pypdf`, `tiktoken` |
| 2 | Pointed to a local `pdfs/` folder | `pathlib.Path` |
| 3 | Extracted text from every PDF page by page | `pypdf.PdfReader` |
| 4 | Chunked docs into 200-token windows, 40-token overlap | `tiktoken` |
| 5 | Converted every chunk to a 1536-dim embedding vector | `text-embedding-3-small` |
| 6 | Stored chunks + vectors in a simple in-memory store | cosine similarity |
| 7 | Embedded the query and retrieved top-3 matching chunks | same embedding model |
| 8 | Fed context + question into GPT → cited answer | `gpt-4o-mini` |
| 9 | Interactive Q&A loop | `input()` |
| 10 | Exported Q&A pairs to JSON | `json` |

**To add more documents:** drop new `.pdf` files into the `pdfs/` folder and re-run the notebook from Step 2.

**Next steps for production:**
- Replace in-memory store with FAISS or Pinecone for large document sets
- Add OCR (`pytesseract`) for scanned / image-only PDFs
- Add re-ranking for higher retrieval precision
- Add streaming for a faster chat-style UI